# HeH⁺ potential energy surface: from exact references to real quantum hardware
**Team:** _Name A_ (algorithm & physics) · _Name B_ (hardware & noise)
**BasQ Qiskit Fall Fest 2026, Challenge 1, Beginner track**

## Abstract
_TODO (both, last 10 min): 4–5 sentences. System, what we pushed beyond the baseline, main numbers (equilibrium length, D_e, best hardware error vs exact), and the main limitation._

## How this notebook scores against the judging criteria
| Criterion | Where we address it |
|---|---|
| **Technical complexity** | §1 qubit tapering 4→2 qubits · §4 larger basis (6-31G), ansatz and optimizer study |
| **Validation** | §2 independent PySCF FCI cross-check · §3 VQE vs exact over the full curve · §6 noisy simulation · §7 real hardware · §8 all levels compared |
| **Hardware awareness** | §5 encoding vs routing depth, noise-aware layout selection · §6 ZNE · §7 dynamical decoupling, twirling, TREX on a real device |
| **Performance and limitations** | §8 error budget (ansatz / shots / noise / mitigation residual) and scaling |
| **Presentation** | Reproducibility below · each section states its claim, evidence and caveat |

## Reproducibility
```bash
pip install -r requirements.txt          # pinned versions
python beginner/check_setup.py           # must print OK
```
Run top to bottom (≈ _TODO_ min without hardware). Hardware results are loaded from `results/hardware_*.csv` unless `RUN_HARDWARE = True`. All energies are **total energies in Hartree**. Chemical accuracy = 1.6 mHa.

<details><summary><b>Team plan (delete before submitting)</b></summary>

| Time | A: algorithm & physics | B: hardware & noise |
|---|---|---|
| 0:00–0:10 | Read the template together, run §0, agree on the distance grid | same |
| 0:10–0:50 | §1, §2, §3 (noiseless PES, equilibrium, dissociation) | §5 transpile study, then **submit the hardware job by ~0:45** (§7) because of queue time |
| 0:50–1:25 | §4 beyond the baseline (6-31G, ansatz/optimizer) | §6 noisy PES and ZNE; collect hardware results when they're ready |
| 1:25–1:45 | §8 error budget and combined figure (A writes it, B supplies numbers) | §7 hardware analysis, §8 noise rows |
| 1:45–2:00 | Abstract and conclusions, **Restart & Run All**, export HTML | Proofread, check figures and job IDs, final commit |

**Git rules for the next two hours:** only edit cells in your own sections. Experiment in `scratch_<name>.ipynb`. Every section saves `results/*.csv` and `figures/*.png`. Pull before you start, and commit and push every ~15 min.
</details>

# 0. Setup

In [ ]:
import time, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from pes_tools import (CHEMICAL_ACCURACY, HARTREE_TO_KCAL_MOL, build_problem,
                       hf_energy, exact_energy, vqe_energy, uccsd_ansatz)
from hardware_tools import (make_mapper, prepare_point, transpile_report, best_transpile,
                            noisy_estimator, estimate, fold_two_qubit_gates, zne_extrapolate,
                            runtime_estimator)

warnings.filterwarnings("ignore", module="scipy")
RESULTS, FIGURES = Path("results"), Path("figures")
RUN_HARDWARE = False  # flip to True only to (re)submit real-device jobs

# Shared distance grid (Angstrom). Agree on this in the first 10 minutes.
DISTANCES = np.round(np.concatenate([np.linspace(0.5, 1.2, 8), np.linspace(1.4, 3.5, 6)]), 3)  # TODO
HW_DISTANCES = [0.6, 0.774, 1.0, 1.5, 2.5]  # TODO: subset for noisy sim / hardware

# 1. System and qubit encoding
> **Owner:** A &nbsp;·&nbsp; **Scores on:** Technical complexity, Presentation &nbsp;·&nbsp; **Timebox:** 0:10–0:20

HeH⁺: 2 electrons (He 2 + H 1 − 1 for the charge), closed-shell singlet. In STO-3G that is 2 spatial orbitals, i.e. 4 spin-orbitals.

**Claim:** exploiting particle-number and ℤ₂ symmetries shrinks the problem from 4 qubits (Jordan-Wigner) to 2 (parity + tapering) with **no loss of accuracy**, and that is what makes the hardware runs in §5–§7 feasible.

_TODO: one short paragraph on why tapering is exact here (symmetry sectors), and a reference._

In [ ]:
problem = build_problem(0.774)
rows = []
for kind in ["jw", "parity", "tapered"]:
    m = make_mapper(problem, kind)
    op = m.map(problem.hamiltonian.second_q_op())
    a = uccsd_ansatz(problem, m)
    rows.append({"encoding": kind, "qubits": op.num_qubits, "pauli_terms": len(op),
                 "uccsd_params": a.num_parameters, "exact_Ha": exact_energy(problem, m)})
encodings = pd.DataFrame(rows); encodings

_TODO (A): 2–3 sentences interpreting the table._

# 2. Classical references and an independent check
> **Owner:** A &nbsp;·&nbsp; **Scores on:** Validation &nbsp;·&nbsp; **Timebox:** 0:20–0:25

Before trusting any quantum number, we check our "exact" reference two independent ways: Qiskit Nature's diagonalisation and PySCF's own FCI solver.

**Pitfall we found:** the tutorial's unfiltered `NumPyMinimumEigensolver` searches every electron count. For HeH⁺ it returns a state with extra electrons, ≈0.16 Ha too low. We filter on particle number and spin (`exact_energy` in `pes_tools.py`).

In [ ]:
from pyscf import gto, scf, fci
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

def pyscf_fci(d, basis="sto3g"):
    mol = gto.M(atom=f"He 0 0 0; H 0 0 {d}", basis=basis, charge=1, spin=0, verbose=0)
    mf = scf.RHF(mol).run()
    return mf.e_tot, fci.FCI(mf).kernel()[0]

e_rhf, e_fci = pyscf_fci(0.774)
p = build_problem(0.774)
unfiltered = GroundStateEigensolver(make_mapper(p, "jw"), NumPyMinimumEigensolver()).solve(p).total_energies[0]
print(f"PySCF RHF          {e_rhf: .8f}   Qiskit HF circuit {hf_energy(p): .8f}")
print(f"PySCF FCI          {e_fci: .8f}   Qiskit exact      {exact_energy(p): .8f}")
print(f"Unfiltered (wrong) {unfiltered: .8f}   off by {unfiltered - e_fci:+.3f} Ha")

# 3. Noiseless potential energy surface
> **Owner:** A &nbsp;·&nbsp; **Scores on:** Validation, Performance &nbsp;·&nbsp; **Timebox:** 0:25–0:50

Noiseless VQE (UCCSD, tapered 2 qubits, SLSQP) against HF and exact over the whole curve. Each point is warm-started from the previous one's parameters, and §8 compares that with cold starts.

In [ ]:
rows, prev = [], None
for d in DISTANCES:
    prob = build_problem(float(d)); m = make_mapper(prob, "tapered")
    v = vqe_energy(prob, mapper=m, initial_point=prev)
    prev = v.optimal_point
    rows.append({"distance_A": d, "hf": hf_energy(prob, m), "vqe": v.energy,
                 "exact": exact_energy(prob, m), "evals": v.cost_function_evals})
pes = pd.DataFrame(rows)
pes["vqe_err"] = (pes.vqe - pes.exact).abs()
pes["correlation"] = pes.exact - pes.hf
pes.to_csv(RESULTS / "pes_noiseless_sto3g.csv", index=False)
pes

In [ ]:
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True, height_ratios=[3, 1])
ax.plot(pes.distance_A, pes.hf, "o--", c="#9e9e9e", label="Hartree-Fock")
ax.plot(pes.distance_A, pes.exact, "-", c="#2ca02c", label="Exact (FCI)")
ax.plot(pes.distance_A, pes.vqe, "x", c="#1f77b4", ms=9, label="VQE (UCCSD, 2 qubits)")
ax.set_ylabel("Total energy (Ha)"); ax.legend(); ax.set_title("HeH$^+$ PES, STO-3G, noiseless")
ax2.semilogy(pes.distance_A, pes.vqe_err, "o-", c="#1f77b4", label="|VQE − exact|")
ax2.axhline(CHEMICAL_ACCURACY, c="red", ls=":", label="chemical accuracy")
ax2.set_xlabel("He–H distance (Å)"); ax2.set_ylabel("Error (Ha)"); ax2.legend()
fig.tight_layout(); fig.savefig(FIGURES / "pes_noiseless_sto3g.png", dpi=150)

### 3a. Equilibrium bond length

In [ ]:
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize_scalar
spl = CubicSpline(pes.distance_A, pes.vqe)
opt = minimize_scalar(spl, bounds=(pes.distance_A.min(), pes.distance_A.max()), method="bounded")
r_eq, e_eq = opt.x, float(opt.fun)
print(f"r_eq (VQE, spline) = {r_eq:.4f} Å,  E = {e_eq:.6f} Ha")
# TODO (A): refine with VQE directly at r_eq; compare to the experimental value (cite it), and comment on basis-set error.

### 3b. Dissociation

In [ ]:
def atom_fci(symbol, charge, spin, basis="sto3g"):
    nelec = {"He": 2, "H": 1}[symbol] - charge
    if nelec == 0:
        return 0.0  # a bare proton has no electrons
    mol = gto.M(atom=f"{symbol} 0 0 0", basis=basis, charge=charge, spin=spin, verbose=0)
    mf = (scf.RHF if spin == 0 else scf.ROHF)(mol).run()
    return fci.FCI(mf).kernel()[0]

channels = {"He + H+": atom_fci("He", 0, 0) + atom_fci("H", 1, 0),
            "He+ + H": atom_fci("He", 1, 1) + atom_fci("H", 0, 1)}
for k, v in channels.items(): print(f"{k:8s} {v: .6f} Ha")
e_inf = min(channels.values())
print(f"Long-distance end of our curve: {pes.exact.iloc[-1]:.6f} Ha")
print(f"D_e ≈ {(e_inf - e_eq):.4f} Ha = {(e_inf - e_eq) * HARTREE_TO_KCAL_MOL:.1f} kcal/mol")
# TODO (A): which channel wins and why (ionisation energies of He vs H)? Is the curve converged at 3.5 Å?

_TODO (A): interpret §3. Where does correlation energy matter most along the curve? Does VQE stay within chemical accuracy everywhere?_

# 4. Beyond the baseline
> **Owner:** A &nbsp;·&nbsp; **Scores on:** Technical complexity, Performance &nbsp;·&nbsp; **Timebox:** 0:50–1:25

Pick the ones that time allows, in this order:
1. **Larger basis (6-31G):** 4 spatial orbitals, 6 tapered qubits, 15 UCCSD parameters. Does basis-set improvement outweigh the ansatz error? Sweep a few distances.
2. **Ansatz comparison:** UCCSD vs a hardware-efficient `efficient_su2` (fewer 2q gates, but needs many more parameters and can get stuck in local minima).
3. **Optimizer comparison:** SLSQP vs COBYLA vs SPSA. SPSA is the realistic choice under shot noise.

In [ ]:
rows = []
for basis in ["sto3g", "6-31g"]:
    for kind in ["jw", "tapered"]:
        t0 = time.perf_counter()
        prob = build_problem(0.774, basis=basis); m = make_mapper(prob, kind)
        v = vqe_energy(prob, mapper=m)
        rows.append({"basis": basis, "encoding": kind, "qubits": v.num_qubits, "params": v.num_parameters,
                     "vqe": v.energy, "exact": exact_energy(prob, m), "seconds": time.perf_counter() - t0})
scaling = pd.DataFrame(rows); scaling["err"] = (scaling.vqe - scaling.exact).abs()
scaling.to_csv(RESULTS / "basis_encoding_scaling.csv", index=False); scaling

In [ ]:
# TODO (A): 6-31G PES on a coarse grid; ansatz/optimizer comparison. Save to results/.

_TODO (A): interpretation._

# 5. Hardware-aware compilation
> **Owner:** B &nbsp;·&nbsp; **Scores on:** Hardware awareness &nbsp;·&nbsp; **Timebox:** 0:10–0:35

**Claim:** the choice of encoding sets the routed circuit depth on a heavy-hex device, and the depth sets the noise. We compare encodings and transpiler optimisation levels, then pick the physical layout with the lowest total reported error (gate + readout errors from the backend calibration), scanning transpiler seeds.

Backend: `FakeFez` (a snapshot of IBM Heron r2 calibration). _TODO (B): switch to the same device family as the real run in §7._

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeFez
backend = FakeFez()

rows = []
for kind in ["jw", "tapered"]:
    pp = prepare_point(0.774, mapping=kind)
    for lvl in [0, 1, 2, 3]:
        r = transpile_report(pp.circuit, backend, optimization_level=lvl, seed=0)
        rows.append({"encoding": kind, "opt_level": lvl, "depth": r["depth"],
                     "two_qubit_gates": r["two_qubit_gates"], "error_score": r["error_score"]})
compile_study = pd.DataFrame(rows)
compile_study.to_csv(RESULTS / "compile_study.csv", index=False); compile_study

In [ ]:
pp_eq = prepare_point(0.774, mapping="tapered")
best = best_transpile(pp_eq.circuit, backend, n_seeds=20)
print("chosen physical qubits:", best["physical_qubits"], "| depth", best["depth"],
      "| 2q gates", best["two_qubit_gates"], "| error score", round(best["error_score"], 4))
best["isa"].draw("mpl", idle_wires=False, fold=-1)
# TODO (B): compare against a deliberately BAD layout (e.g. initial_layout on the worst qubits) to show it matters.

_TODO (B): interpretation. Depth/2q gates for JW vs tapered, and why the chosen qubits won._

# 6. Noisy simulation and error mitigation
> **Owner:** B &nbsp;·&nbsp; **Scores on:** Validation, Hardware awareness &nbsp;·&nbsp; **Timebox:** 0:50–1:25

We optimise parameters noiselessly (§3), then **evaluate** them under the backend's noise model. That separates noise error from optimisation error. ZNE: fold each 2-qubit gate (scale factors 1, 3, 5) and extrapolate to zero noise.

In [ ]:
est = noisy_estimator(backend)
SCALES = [1, 3, 5]
rows = []
for d in HW_DISTANCES:
    pp = prepare_point(d, mapping="tapered")
    isa = best_transpile(pp.circuit, backend, n_seeds=10)["isa"]
    ev, sd = estimate([fold_two_qubit_gates(isa, s) for s in SCALES], pp.qubit_op, est, precision=0.002)
    E = ev + pp.nuclear_repulsion
    rows.append({"distance_A": d, "exact": pp.e_exact, "vqe_noiseless": pp.e_vqe,
                 "noisy": E[0], "noisy_std": sd[0],
                 "zne_linear": zne_extrapolate(SCALES, E, 1), "zne_quad": zne_extrapolate(SCALES, E, 2),
                 **{f"scale_{s}": e for s, e in zip(SCALES, E)}})
noisy = pd.DataFrame(rows)
for c in ["noisy", "zne_linear", "zne_quad"]:
    noisy[f"{c}_err"] = (noisy[c] - noisy.exact).abs()
noisy.to_csv(RESULTS / "pes_noisy_sim.csv", index=False); noisy

In [ ]:
# TODO (B): plot noisy vs ZNE vs exact; also an energy-vs-scale plot at one distance showing the extrapolation.
# Extra: repeat for the JW encoding at r_eq to show how much worse the deeper circuit is.

_TODO (B): interpretation. Is ZNE within chemical accuracy? Is linear or quadratic better, and does the extrapolation amplify shot noise (error bars)?_

# 7. Real quantum hardware
> **Owner:** B &nbsp;·&nbsp; **Scores on:** Validation, Hardware awareness &nbsp;·&nbsp; **Timebox:** submit by ~0:45, analyse 1:25–1:45

**Submit early:** queue times are unpredictable. Send **all distances × all ZNE scales as one job** (a single list of PUBs), which needs only one queue wait.

Error suppression and mitigation used: dynamical decoupling (XpXm), Pauli twirling on gates and measurements, TREX readout mitigation (`resilience_level=1`), plus our own folding-based ZNE on top.

Set your IBM Quantum account up once in a terminal (keep the token out of the notebook):
```python
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="...", instance="...")
```

In [ ]:
HW_FILE = RESULTS / "hardware_pes.csv"
if RUN_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    hw_backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
    print("Running on", hw_backend.name)
    points = [prepare_point(d, mapping="tapered") for d in HW_DISTANCES]
    pubs, meta = [], []
    for pp in points:
        isa = best_transpile(pp.circuit, hw_backend, n_seeds=20)["isa"]
        for s in SCALES:
            c = fold_two_qubit_gates(isa, s)
            pubs.append((c, pp.qubit_op.apply_layout(c.layout))); meta.append((pp, s))
    job = runtime_estimator(hw_backend).run(pubs)
    print("job id:", job.job_id())  # paste into the appendix
    res = job.result()
    rows = []
    for d in HW_DISTANCES:
        idx = [i for i, (pp, s) in enumerate(meta) if pp.distance == d]
        E = np.array([float(res[i].data.evs) for i in idx]) + meta[idx[0]][0].nuclear_repulsion
        pp = meta[idx[0]][0]
        rows.append({"distance_A": d, "exact": pp.e_exact, "hw_raw": E[0],
                     "hw_zne_linear": zne_extrapolate(SCALES, E, 1), "backend": hw_backend.name,
                     "job_id": job.job_id(), **{f"scale_{s}": e for s, e in zip(SCALES, E)}})
    pd.DataFrame(rows).to_csv(HW_FILE, index=False)

hardware = pd.read_csv(HW_FILE) if HW_FILE.exists() else None
hardware

In [ ]:
# TODO (B): hardware vs noisy-sim vs exact plot; errors vs chemical accuracy.

_TODO (B): interpretation. Does the noisy simulator predict hardware well? What did each mitigation layer buy?_

# 8. Performance and limitations
> **Owner:** A (writes) + B (numbers) &nbsp;·&nbsp; **Scores on:** Performance and limitations &nbsp;·&nbsp; **Timebox:** 1:25–1:45

### Error budget at equilibrium
| Source | Size (Ha) | Evidence |
|---|---|---|
| Basis set (STO-3G vs 6-31G) | _TODO_ | §4 |
| Ansatz / optimiser (VQE vs exact, noiseless) | _TODO_ | §3 |
| Shot noise (std at chosen precision) | _TODO_ | §6 |
| Gate and readout noise (raw noisy − noiseless) | _TODO_ | §6 / §7 |
| Residual after mitigation (ZNE − exact) | _TODO_ | §6 / §7 |

### Combined figure
_TODO: one plot with exact, noiseless VQE, noisy sim (raw and ZNE) and hardware (raw and ZNE) vs distance. This is the headline figure._

In [ ]:
# TODO: combined headline figure from results/*.csv

### Limitations and scaling
_TODO (both): be honest and specific. For example:_
- _Fixed-parameter evaluation: we never ran the optimisation loop on hardware. What would change with SPSA under noise?_
- _Tapering makes HeH⁺/STO-3G trivially small; how do qubits and 2q gates grow with basis (use §4 numbers)?_
- _ZNE: assumptions (noise scales linearly with folding), variance amplification, and cost (3× circuits)._
- _A noise-aware layout uses calibration data that drifts._

# 9. Conclusions
> **Owner:** both &nbsp;·&nbsp; **Scores on:** Presentation &nbsp;·&nbsp; **Timebox:** 1:45–2:00

_TODO: 4–6 bullets with numbers. Then 2–3 bullets on what we'd do with more time._

# Appendix
- **Hardware:** backend _TODO_, job ID(s) _TODO_, date _TODO_
- **Environment:** see the cell below.
- **Credits:** builds on B. Tirado, *Ground-State Energy of a Molecule with VQE*, BasQ Qiskit Fall Fest 2026 tutorial (CC BY 4.0 / Apache 2.0).

In [ ]:
from importlib.metadata import version
import sys
print("python", sys.version.split()[0])
for p in ["qiskit", "qiskit-nature", "qiskit-algorithms", "qiskit-aer", "qiskit-ibm-runtime", "pyscf"]:
    print(f"{p:20s}{version(p)}")